Подключение диска

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install keras

Импортирование необхрдимых библиотек

In [ ]:
import random
from pathlib import Path
import os
import librosa

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import torchvision
from torchvision import transforms

from PIL import Image
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split, ConcatDataset
from tqdm.notebook import tqdm

from torch import nn
from torchsummary import summary
import torch.nn.functional as F

from keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from glob import glob


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

Задание путей

In [ ]:
train_path = "/content/drive/MyDrive/datasets/Chords/Training"
test_path = "/content/drive/MyDrive/datasets/Chords/Test"

Константы работы со звуком

In [ ]:
sr = 16000
dt = 1.0
N_CLASSES = 6
n_mels = 64
hop_length = 256
n_fft = 1024
threshold=0.01
frames = (sr//hop_length)+1

Обработка звука

In [ ]:
def envelope(y):
    mask = []
    y1 = pd.Series(y).apply(np.abs)
    y_mean = y1.rolling(window=int(sr/20),
                       min_periods=1,
                       center=True).max()
    for mean in y_mean:
        if mean > threshold:
            mask.append(True)
        else:
            mask.append(False)
    return mask

In [ ]:
def load_audio(path, sr):
    rate = librosa.get_samplerate(path)
    y1, _ = librosa.load(path, sr=rate)
    wav = librosa.resample(y=y1, orig_sr=rate, target_sr=sr)
    return wav

In [ ]:
def get_melspectogram(waveform):
    mel_spectrogram = librosa.feature.melspectrogram(y=waveform, sr=sr, n_fft=n_fft,
                                                     hop_length=hop_length, n_mels=n_mels)
    log_mel_spectrogram = librosa.power_to_db(mel_spectrogram)
    return log_mel_spectrogram

In [ ]:
class AudDataset:
    def __init__(self, path, n_classes=6):
        self.path = path
        self.sr = sr
        self.dt = dt
        self.n_mels = n_mels
        self.frames = frames
        self.n_classes = n_classes

        self.wav_paths = glob('{}/**'.format(self.path), recursive=True)
        self.wav_paths = [x.replace(os.sep, '/') for x in self.wav_paths if '.wav' in x]

        self.classes = sorted(os.listdir(self.path))
        self.le = LabelEncoder()
        self.le.fit(self.classes)

        self.labels = [os.path.split(x)[0].split('/')[-1] for x in self.wav_paths]
        self.labels = self.le.transform(self.labels)

        self.X, self.Y = self._prepare_data()

    def _prepare_data(self):
        X = np.empty((1, int(self.sr * self.dt)), dtype=np.float32)
        Y = []

        for file, label in zip(self.wav_paths, self.labels):
            y1 = load_audio(file, self.sr)
            mask = envelope(y1)
            y = y1[mask]
            n = y.shape[0]
            m = n // self.sr
            for k in range(m):
                a = np.expand_dims(y[k * self.sr:(k + 1) * self.sr], axis=0)
                X = np.append(X, a, axis=0)
                Y.append(label)

        X = np.delete(X, 0, axis=0)
        Y = np.array(Y, dtype=np.int32)

        n_samples = X.shape[0]
        X_mel = np.empty((n_samples, self.n_mels, self.frames), dtype=np.float32)

        for i in np.arange(n_samples):
            X_mel[i] = get_melspectogram(X[i])

        return X_mel, Y

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

    def __len__(self):
        return len(self.X)

In [ ]:
test = AudDataset(path=test_path)
train = AudDataset(path=train_path)

In [ ]:
train_dataloader = DataLoader(train, batch_size=32, shuffle=True)
valid_dataloader = DataLoader(test, batch_size=32, shuffle=False)

Архитектура аудиомодели

In [ ]:
class Conv2DModel(nn.Module):
    def __init__(self, n_classes):
        super(Conv2DModel, self).__init__()

        # Нормализация слоев
        self.layer_norm = nn.LayerNorm([n_mels, frames])

        # Сверточные слои
        self.conv1 = nn.Conv2d(1, 8, kernel_size=(7, 7), padding='same')
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2), padding=0)

        self.conv2 = nn.Conv2d(8, 16, kernel_size=(5, 5), padding='same')
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2), padding=0)

        self.conv3 = nn.Conv2d(16, 16, kernel_size=(3, 3), padding='same')
        self.pool3 = nn.MaxPool2d(kernel_size=(2, 2), padding=0)

        self.conv4 = nn.Conv2d(16, 32, kernel_size=(3, 3), padding='same')
        self.pool4 = nn.MaxPool2d(kernel_size=(2, 2), padding=0)

        self.conv5 = nn.Conv2d(32, 32, kernel_size=(3, 3), padding='same')

        # Полносвязные слои
        self.dropout = nn.Dropout(0.2)
        self.fc1 = nn.Linear(32 * (n_mels // 16) * (frames // 16), 64)
        self.fc2 = nn.Linear(64, n_classes)

    def forward(self, x):
        # Применение LayerNorm к формату последнего канала и восстановление формата первого канала
        x = self.layer_norm(x.squeeze(1)).unsqueeze(1)

        x = torch.tanh(self.conv1(x))
        x = self.pool1(x)

        x = F.relu(self.conv2(x))
        x = self.pool2(x)

        x = F.relu(self.conv3(x))
        x = self.pool3(x)

        x = F.relu(self.conv4(x))
        x = self.pool4(x)

        x = F.relu(self.conv5(x))

        x = torch.flatten(x, start_dim=1)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return F.softmax(x, dim=1)


model = Conv2DModel(n_classes=6).to(device)

summary(model, (64, 63))

Подготовка к обучению модели

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
criterion = nn.CrossEntropyLoss()


train_losses = []
val_losses = []

train_acc_scores = []
val_acc_scores = []

best_val_acc = 0.0
best_model_path = '/content/drive/MyDrive/best_model_aud.pth' # Сохраняет модель на диск, что бы не потерять при закончившихся ресурсах колаба

num_epochs = 50

Цикл обучения модели

In [ ]:
for epoch in range(num_epochs):
    scheduler.step()
    model.train()
    running_loss = 0.0
    train_true = []
    train_pred = []

    for batch in tqdm(train_dataloader):
        inputs, labels = batch
        inputs = torch.tensor(inputs)
        labels = torch.tensor(labels)
        labels = labels.type(torch.LongTensor)
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        train_true.extend(labels.cpu().numpy())
        train_pred.extend(preds.cpu().numpy())

    train_acc = accuracy_score(train_true, train_pred)
    train_losses.append(running_loss / len(train_dataloader))
    train_acc_scores.append(train_acc)


    model.eval()
    val_running_loss = 0.0
    val_true = []
    val_pred = []

    # валидационный цикл, когда мы оцениваем качество работы модели на отложенной выборке
    with torch.no_grad():
        for batch in tqdm(valid_dataloader):
            inputs, labels = batch
            inputs = torch.tensor(inputs)
            labels = torch.tensor(labels)
            labels = labels.type(torch.LongTensor)
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            val_true.extend(labels.cpu().numpy())
            val_pred.extend(preds.cpu().numpy())

    val_acc = accuracy_score(val_true, val_pred)
    val_losses.append(val_running_loss / len(valid_dataloader))
    val_acc_scores.append(val_acc)

    # если получившаяся модель лучше предыдущей, сохраним чекпоинт
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f'New best model saved with Accuracy: {best_val_acc:.4f}')


    # выведем в консоль получившиеся результаты на отдельной эпохе
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {train_losses[-1]:.4f}, Train Accuracy: {train_acc:.4f}, '
          f'Val Loss: {val_losses[-1]:.4f}, Val Accuracy: {val_acc:.4f}')

Рисование графиков

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_acc_scores, label='Train Accuracy')
plt.plot(val_acc_scores, label='Validation Accuracy')
plt.title('Accuracy Score')
plt.xlabel('Epoch')
plt.ylabel('Accuracy Score')
plt.legend()

plt.show()